# Prepare CHP data

This prepares yearly CHP generation in MWh per country based on the Eurostat CHP questionaire from https://ec.europa.eu/eurostat/documents/38154/4956229/CHPdata2005-2017.xlsx/

- per technology shares are calculated using OPSD data where available

- For other countries, we create shares based on yearly entsoe generation data from parse_generation_entsoe_sftp

- Furthermore, we prepare yearly CHP profile from original data (lion's?)

Careful: '../parsed_data/generation_'+year+'_annual_entsoe.csv' is needed for 2015-2017!

In [1]:
import pandas as pd
import numpy as np
import wget
import os

In [2]:
download = "no"

In [3]:
dir_in = "../source_data/chp/"
dir_out = "../parsed_data/"
years = ['2015','2016','2017'] #careful if extending list of years as SK 2014 is missing
fn = "CHPdata2005-2017.xlsx"
fn_heat_dem = "../source_data/chp/chp_data.xlsx"
fn_opsd = "conventional_power_plants_EU.csv"
url = "https://ec.europa.eu/eurostat/documents/38154/4956229/CHPdata2005-2017.xlsx/871cc151-5733-423f-ae38-de9b733aa81e"
url_opsd = 'https://data.open-power-system-data.org/conventional_power_plants/2020-10-01/conventional_power_plants_EU.csv'

In [4]:
#only if data has to be downloaded again:
if download == "yes":
    os.remove(dir_in+fn)
    wget.download(url,dir_in+fn)

! if data is downloaded again, sheet 2017 needs "2017" in cell A3, otherwise, 2017 values are missing

! should also copy all PJ data to row 37 to be consistent

In [5]:
map_country_ISO = {
	"Austria" : "AT",
	"Belgium" : "BE",
	"Belgium1" : "BE",
	"Bulgaria" : "BG",
	"Croatia" : "HR",
	"Cyprus" : "CY",
	"Czech" : "CZ",
	"Czech Republic" : "CZ",
	"Czechia" : "CZ",
	"Denmark" : "DK",
	"Estonia" : "EE",
	"Estonia2" : "EE",
	"Finland" : "FI",
	"France" : "FR",
	"Germany" : "DE",
	"Germany1" : "DE",
	"Germany1, 2" : "DE",
	"Greece" : "GR",
	"Greece2" : "GR",
	"Hungary" : "HU",
	"Hungary1" : "HU",
	"Hungary3" : "HU",
	"Ireland" : "IE",
	"Ireland1" : "IE",
	"Italy" : "IT",
	"Latvia" : "LV",
	"Lithuania" : "LT",
	"Luxembourg" : "LU",
	"Malta" : "MT",
	"Netherlands" : "NL",
	"Norway" : "NO",
	"Norway2" : "NO",
	"Poland" : "PL",
	"Portugal" : "PT",
	"Portugal1" : "PT",
	"Romania" : "RO",
	"Slovakia" : "SK",
	"Slovakia3" : "SK",
	"Slovenia" : "SI",
	"Slovenia1" : "SI",
	"Spain" : "ES",
	"Sweden" : "SE",
	"Sweden2" : "SE",
	"United Kingdom" : "GB",
    "Estonia*" : "EE"
}

In [6]:
dict_opsd_tech = {
    'Natural gas':'Gas',
    'Oil':'Oil',
    'Biomass and biogas':'Biomass',
    'Nuclear':'Nuclear',
    'Non-renewable waste':'Other',
    'Mixed fossil fuels':'Other',
    'Hard coal':'HardCoal',
    'Other or unspecified energy sources':'Other',
    'Lignite':'Lignite',
    'Bioenergy':'Biomass',
    'Other fossil fuels':'Other',
    'Other fuels':'Other',
    'Waste':'Other'
}

In [7]:
#opsd only has chp data for the following countries
opsd_countries = ['BE', 'FI', 'ES', 'SE', 'SI', 'AT', 'DE']

In [8]:
#lame fix for column naming:
year_to_country = {
    2015 : "country",
    2016 : "country",
    2017 : "country",
}

In [9]:
df=pd.DataFrame()
for year in years:
    df_temp = pd.read_excel(dir_in + fn, sheet_name = year,
                            header=2, usecols = "A:E", nrows=30,na_values=":") 
    df_temp = df_temp.rename(columns = year_to_country)
    df_temp['year'] = year
    df = df.append(df_temp, ignore_index=True)

Rename countries to ISO and remove non listed countries or regions

In [10]:
df.index = df['country']
df = df.rename(index=map_country_ISO)
df = df[df['country'].isin(map_country_ISO)].drop(columns='country').reset_index()

Convert CHP generation to MWh and drop non needed columns

In [11]:
df['CHP_MWh'] = df['CHP electricity generation, TWh']*1000*1000
df_chp_gen = df[['year','country','CHP_MWh']].copy().set_index(['year','country'])

now we copy 2016 data for NO as this is missing in 2017

In [12]:
df_chp_gen.loc[('2017','NO'),:] = df_chp_gen.loc[('2016','NO'),:]

In [13]:
df_chp_gen.head(1)

,,CHP_MWh
year,country,
2015,BE,12479000.0


Now we load OPSD data to calculate shares per technology

In [14]:
if download == "yes":
    wget.download(url_opsd,dir_in+fn_opsd)

In [15]:
df_opsd = pd.read_csv(dir_in+fn_opsd)
df_opsd['chp_capacity'] = df_opsd['capacity'][df_opsd['chp']=="Yes"]
df_opsd['chp_capacity'] = df_opsd['capacity'][df_opsd['chp']=="yes"]
df_opsd['tech'] = df_opsd.energy_source.map(dict_opsd_tech)
df_opsd.head()

,name,company,street,postcode,city,country,capacity,energy_source,technology,chp,...,lon,eic_code,energy_source_level_1,energy_source_level_2,energy_source_level_3,additional_info,comment,source,chp_capacity,tech
0,Marcinelle Energie (Carsid),DIRECT ENERGIE,NaN,NaN,NaN,BE,413.0,Natural gas,Combined cycle,NaN,...,4.40645,22WMARCIN000179H,Fossil fuels,Natural gas,NaN,NaN,NaN,https://www.elia.be/en/grid-data/power-generat...,NaN,Gas
1,Aalst Syral GT,Electrabel,NaN,NaN,NaN,BE,43.0,Natural gas,Gas turbine,Yes,...,NaN,NaN,Fossil fuels,Natural gas,NaN,NaN,NaN,https://www.elia.be/en/grid-data/power-generat...,NaN,Gas
2,Aalst Syral ST,Electrabel,NaN,NaN,NaN,BE,5.0,Natural gas,Steam turbine,Yes,...,NaN,NaN,Fossil fuels,Natural gas,NaN,NaN,NaN,https://www.elia.be/en/grid-data/power-generat...,NaN,Gas
3,AALTER TJ,Electrabel,NaN,NaN,NaN,BE,18.0,Oil,Gas turbine,NaN,...,NaN,NaN,Fossil fuels,Oil,NaN,NaN,NaN,https://www.elia.be/en/grid-data/power-generat...,NaN,Oil
4,Amercoeur 1 R TGV,Electrabel,NaN,NaN,NaN,BE,451.0,Natural gas,Combined cycle,NaN,...,4.39518,22WAMERCO000010Y,Fossil fuels,Natural gas,NaN,NaN,NaN,https://www.elia.be/en/grid-data/power-generat...,NaN,Gas


Extract chp technologies

In [16]:
chp_techs = df_opsd.tech[df_opsd['chp_capacity'] > 0].unique()
chp_techs

array(['HardCoal', 'Biomass', 'Other', 'Gas', 'Lignite', 'Oil'],
      dtype=object)

In [17]:
df_opsd_chp_cap =  df_opsd.groupby(['country']).sum()[['chp_capacity']]
df_opsd_chp_cap.head()

,chp_capacity
country,
AT,4562.2000
BE,0.0000
CH,0.0000
CZ,0.0000
DE,58718.0049


In [18]:
df_opsd_agg = df_opsd.groupby(['country','tech']).sum()[['capacity','chp_capacity']]
df_opsd_agg = df_opsd_agg.reset_index().merge(df_opsd_chp_cap.reset_index(),how='left',on='country')
df_opsd_agg['chp_share'] = df_opsd_agg['chp_capacity_x'] / df_opsd_agg['chp_capacity_y']
df_opsd_agg = df_opsd_agg.set_index(['country','tech'])[['chp_share']]
df_opsd_agg = df_opsd_agg[df_opsd_agg['chp_share']>0]
df_opsd_agg = df_opsd_agg.reset_index().pivot_table(index='country',columns='tech',values='chp_share').fillna(0)
df_opsd_agg.head()

tech,Biomass,Gas,HardCoal,Lignite,Oil,Other
country,,,,,,
AT,0.000000,0.946079,0.000000,0.000000,0.000000,0.053921
DE,0.011587,0.294515,0.346726,0.269764,0.016223,0.061184
ES,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
SE,0.226453,0.260521,0.130261,0.000000,0.000000,0.382766
SI,0.000000,0.072227,0.000000,0.927773,0.000000,0.000000


now merge the two dfs

In [19]:
df_chp_tech = df_chp_gen.reset_index().merge(df_opsd_agg,how='left',on='country').fillna(0)
df_chp_tech['sum'] = df_chp_tech['Biomass'] + df_chp_tech['Gas'] + df_chp_tech['HardCoal'] + df_chp_tech['Lignite'] + df_chp_tech['Oil'] + df_chp_tech['Other']
df_chp_tech.head()

,year,country,CHP_MWh,Biomass,Gas,HardCoal,Lignite,Oil,Other,sum
0,2015,BE,12479000.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
1,2015,BG,2945000.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
2,2015,CZ,11785000.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3,2015,DK,11568000.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
4,2015,DE,78787000.0,0.011587,0.294515,0.346726,0.269764,0.016223,0.061184,1.0


eventually, for countries where chp capacities are unknown, we calculate shares per technology based on their share in yearly generation

In [20]:
df_gen = pd.DataFrame()
for year in years:
    df_gen_temp = pd.read_csv('../parsed_data/generation_'+year+'_annual_entsoe.csv')
    df_gen_temp['year'] = year
    df_gen = df_gen.append(df_gen_temp)
df_gen = df_gen[df_gen.tech.isin(chp_techs)]
df_gen = df_gen.groupby(['year','country','tech']).sum()[['net_generation']]
df_gen.head()

net_generation
year country tech                    
2015 AT      Biomass         2.411676
             Gas             7.641010
             HardCoal        1.675485
             Oil             0.000000
             Other           1.994389

In [21]:
df_gen_total = df_gen.reset_index().groupby(['year','country']).sum()
df_gen_total.head()

net_generation
year country                
2015 AT            13.722561
     BE            29.740867
     BG            22.578594
     CH             0.024822
     CY             1.666379

In [22]:
df_gen_shares = df_gen.reset_index().merge(df_gen_total.reset_index(), how='left', on=['year','country'])
df_gen_shares['chp_share'] = df_gen_shares['net_generation_x'] / df_gen_shares['net_generation_y']
df_gen_shares = df_gen_shares[['year','country','tech','chp_share']]
df_gen_shares = df_gen_shares.pivot_table(index=['year','country'],columns='tech',values='chp_share')
df_gen_shares.head()

tech           Biomass       Gas  HardCoal   Lignite       Oil     Other
year country                                                            
2015 AT       0.175745  0.556821  0.122097       NaN  0.000000  0.145337
     BE       0.074476  0.644153  0.062399  0.000000  0.000118  0.218853
     BG       0.007635       NaN       NaN  0.992365       NaN       NaN
     CH            NaN  1.000000       NaN       NaN       NaN       NaN
     CY            NaN       NaN       NaN       NaN  1.000000       NaN

now merge this into chp df, calculate the yearly CHP generation per technology and clean up

In [23]:
df_chp_tech_merged = df_chp_tech.merge(df_gen_shares, how='left',on=['year','country'])
df_chp_tech_merged['HardCoal_x'][df_chp_tech_merged['sum'] == 0] = df_chp_tech_merged['HardCoal_y']
df_chp_tech_merged['Biomass_x'][df_chp_tech_merged['sum'] == 0] = df_chp_tech_merged['Biomass_y']
df_chp_tech_merged['Other_x'][df_chp_tech_merged['sum'] == 0] = df_chp_tech_merged['Other_y']
df_chp_tech_merged['Gas_x'][df_chp_tech_merged['sum'] == 0] = df_chp_tech_merged['Gas_y']
df_chp_tech_merged['Lignite_x'][df_chp_tech_merged['sum'] == 0] = df_chp_tech_merged['Lignite_y']
df_chp_tech_merged['Oil_x'][df_chp_tech_merged['sum'] == 0] = df_chp_tech_merged['Oil_y']
df_chp_tech_merged = df_chp_tech_merged[['year','country','CHP_MWh','HardCoal_x', 'Biomass_x', 'Other_x', 'Gas_x', 'Lignite_x', 'Oil_x']].set_index(['year','country'])
df_chp_tech_merged = df_chp_tech_merged.rename(columns={'HardCoal_x':'HardCoal', 'Biomass_x':'Biomass', 'Other_x':'Other', 'Gas_x':'Gas', 'Lignite_x':'Lignite', 'Oil_x':'Oil'})
df_chp_tech_merged.head()

<ipython-input-23-ca47114cefbd>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_chp_tech_merged['HardCoal_x'][df_chp_tech_merged['sum'] == 0] = df_chp_tech_merged['HardCoal_y']
<ipython-input-23-ca47114cefbd>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_chp_tech_merged['Biomass_x'][df_chp_tech_merged['sum'] == 0] = df_chp_tech_merged['Biomass_y']
<ipython-input-23-ca47114cefbd>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


CHP_MWh  HardCoal   Biomass     Other       Gas   Lignite  \
year country                                                                 
2015 BE       12479000.0  0.062399  0.074476  0.218853  0.644153  0.000000   
     BG        2945000.0       NaN  0.007635       NaN       NaN  0.992365   
     CZ       11785000.0  0.105163  0.049556  0.109573  0.036204  0.698400   
     DK       11568000.0  0.557477  0.028039  0.112096  0.286610       NaN   
     DE       78787000.0  0.346726  0.011587  0.061184  0.294515  0.269764   

                   Oil  
year country            
2015 BE       0.000118  
     BG            NaN  
     CZ       0.001104  
     DK       0.015778  
     DE       0.016223

In [24]:
df_chp_tech_merged['HardCoal'] = df_chp_tech_merged['HardCoal']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged['Biomass'] = df_chp_tech_merged['Biomass']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged['Other'] = df_chp_tech_merged['Other']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged['Gas'] = df_chp_tech_merged['Gas']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged['Lignite'] = df_chp_tech_merged['Lignite']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged['Oil'] = df_chp_tech_merged['Oil']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged = df_chp_tech_merged.drop(columns='CHP_MWh')
df_chp_tech_merged.head()

HardCoal        Biomass         Other           Gas  \
year country                                                            
2015 BE       7.786786e+05  929388.799836  2.731070e+06  8.038386e+06   
     BG                NaN   22484.609981           NaN           NaN   
     CZ       1.239344e+06  584012.763344  1.291316e+06  4.266647e+05   
     DK       6.448894e+06  324360.823677  1.296724e+06  3.315504e+06   
     DE       2.731752e+07  912875.284967  4.820527e+06  2.320398e+07   

                   Lignite           Oil  
year country                              
2015 BE       0.000000e+00  1.476973e+03  
     BG       2.922515e+06           NaN  
     CZ       8.230647e+06  1.301610e+04  
     DK                NaN  1.825173e+05  
     DE       2.125392e+07  1.278185e+06

In [25]:
df_chp_tech_long = df_chp_tech_merged.pivot_table(index=['year','country'], values=['HardCoal', 'Biomass', 'Other', 'Gas', 'Lignite', 'Oil'])
df_chp_tech_long.head()

Biomass           Gas      HardCoal       Lignite  \
year country                                                            
2015 AT            0.000000  8.518492e+06  0.000000e+00  0.000000e+00   
     BE       929388.799836  8.038386e+06  7.786786e+05  0.000000e+00   
     BG        22484.609981           NaN           NaN  2.922515e+06   
     CY                 NaN           NaN           NaN           NaN   
     CZ       584012.763344  4.266647e+05  1.239344e+06  8.230647e+06   

                       Oil         Other  
year country                              
2015 AT           0.000000  4.855079e+05  
     BE        1476.972876  2.731070e+06  
     BG                NaN           NaN  
     CY        3980.000000           NaN  
     CZ       13016.100025  1.291316e+06

In [26]:
df_chp_tech_long = df_chp_tech_merged.reset_index().melt(id_vars=['year','country'],
                                           var_name="tech",
                                           value_name="MWh")
df_chp_tech_long.head()

,year,country,tech,MWh
0,2015,BE,HardCoal,7.786786e+05
1,2015,BG,HardCoal,NaN
2,2015,CZ,HardCoal,1.239344e+06
3,2015,DK,HardCoal,6.448894e+06
4,2015,DE,HardCoal,2.731752e+07


## Now we also create hourly profiles

In [27]:
df_heat_demand_in = pd.read_excel(fn_heat_dem)
df_heat_demand_in.head(1)

,date,heat_demand
0,2014-01-01 00:00:00+00:00,0.86235


In [28]:
df_heat_demand = df_heat_demand_in.drop("date", axis = 1)
df_heat_demand["heat_demand_relative"] = df_heat_demand["heat_demand"]/df_heat_demand["heat_demand"].sum()
df_heat_demand.tail()

,heat_demand,heat_demand_relative
8755,0.896596,0.000153
8756,1.000000,0.000171
8757,1.000000,0.000171
8758,1.000000,0.000171
8759,1.000000,0.000171


# Export both to csv

In [29]:
df_chp_tech_long.to_csv(dir_out + "chp_generation.csv", encoding="utf-8", index=False)

In [30]:
df_heat_demand.to_csv(dir_out + "heat_demand.csv", encoding="utf-8", index = False)

In [31]:
df_chp_tech_long[(df_chp_tech_long.country=='DE') & (df_chp_tech_long.tech=='Gas')]

,year,country,tech,MWh
265,2015,DE,Gas,2.320398e+07
294,2016,DE,Gas,2.589820e+07
323,2017,DE,Gas,2.778949e+07


In [32]:
df_gen = df_gen.reset_index()
test = df_gen[(df_gen.country=='DE') & (df_gen.tech=='Gas')] .merge(df_chp_tech_long[(df_chp_tech_long.country=='DE') & (df_chp_tech_long.tech=='Gas')], on= ['year','country','tech'])

In [39]:
test['test'] = test['net_generation'] - (test['MWh']/1000000)
test.head()

,year,country,tech,net_generation,MWh,test
0,2015,DE,Gas,13.473781,2.320398e+07,-9.730197
1,2016,DE,Gas,23.455237,2.589820e+07,-2.442967
2,2017,DE,Gas,25.890947,2.778949e+07,-1.898546
